# Experimentos del proyecto: fallas de SSD en MLflow

Registra en MLflow, en el experimento `fallas-ssd`, una corrida por cada modelo de la Semana 10: línea base, modelo lineal, ensamblaje y red neuronal. La preparación y la división salen de `pipeline_proyecto.py`, sin copiar su código.

Métrica: **average precision** (área bajo la curva precisión-recall). Solo el 5,94 % de los SSD falla, así que la exactitud de "nunca falla" ya es 94 %. La average precision mide qué tan arriba quedan las fallas al ordenar los discos por riesgo, y su piso es la proporción de fallas.

Los datos salen de `../ingesta/preparar_datos.py`, que los deja en `../data/processed/ssd_mc1.csv`.

In [1]:
import hashlib
import logging
import time
import uuid

import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from pipeline_proyecto import ARCHIVO, cargar_datos, dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

X_ent, X_prueba, y_ent, y_prueba = dividir(cargar_datos())
print(f"Entrenamiento: {X_ent.shape}  Prueba: {X_prueba.shape}  tasa de falla {y_ent.mean():.4f}")

Entrenamiento: (141520, 14)  Prueba: (35380, 14)  tasa de falla 0.0594


In [2]:
def huella(ruta):
    """Los primeros 12 caracteres del MD5 del archivo: si cambia un byte del
    CSV, la huella cambia."""
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(ARCHIVO)
# Cada ejecucion completa del notebook lleva su propia etiqueta: la tabla del
# final sale solo de las corridas de esta ejecucion, aunque el experimento
# guarde corridas de pruebas anteriores.
ejecucion = uuid.uuid4().hex[:8]
print("Huella de ssd_mc1.csv:", huella_datos, "  ejecucion:", ejecucion)

Huella de ssd_mc1.csv: ef33432c8970   ejecucion: c9fd80c0


In [3]:
# Sin MLFLOW_TRACKING_URI (fuera del Codespace), MLflow guarda el registro al
# lado del notebook. Esa carpeta no se sube a git.
print("Registro de MLflow:", mlflow.get_tracking_uri())
EXPERIMENTO = "fallas-ssd"
mlflow.set_experiment(EXPERIMENTO)

Registro de MLflow: file:///C:/Users/AGES/Documents/fundamentos-ciencia-datos/data-science-end-to-end/psets/proyecto/modelado/mlruns


<Experiment: artifact_location='file:///C:/Users/AGES/Documents/fundamentos-ciencia-datos/data-science-end-to-end/psets/proyecto/modelado/mlruns/564609671141715231', creation_time=1794248831270, experiment_id='564609671141715231', last_update_time=1794248831270, lifecycle_stage='active', name='fallas-ssd', tags={}>

In [4]:
SEMILLA_DIVISION = 42
corridas = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion, mide la average precision
    en entrenamiento y en prueba, y registra la corrida en MLflow."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre) as corrida:
        parametros = {clave: valor for clave, valor in modelo.get_params().items()}
        parametros.update({
            "modelo": type(modelo).__name__,
            "division": "train_test_split estratificado, test_size=0.2",
            "semilla_division": SEMILLA_DIVISION,
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
        })
        mlflow.log_params(parametros)
        mlflow.set_tag("ejecucion", ejecucion)

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        prob_ent = pipeline.predict_proba(X_ent)[:, 1]
        prob_prueba = pipeline.predict_proba(X_prueba)[:, 1]
        metricas = {
            "ap_entrenamiento": average_precision_score(y_ent, prob_ent),
            "ap_prueba": average_precision_score(y_prueba, prob_prueba),
            "auc_prueba": roc_auc_score(y_prueba, prob_prueba),
            "segundos": segundos,
        }
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(pipeline, "modelo")

    corridas[nombre] = corrida.info.run_id
    print(f"{nombre:28s} entrenamiento {metricas['ap_entrenamiento']:.4f}   prueba {metricas['ap_prueba']:.4f}   {segundos:6.1f} s")

## Los cuatro modelos de la Semana 10

In [5]:
# Linea base: siempre la proporcion de fallas. No ordena a los discos.
evaluar("linea base", DummyClassifier(strategy="prior"))
# Lineal: regresion logistica con la regularizacion por defecto.
evaluar("logistica C=1", LogisticRegression(C=1.0, max_iter=2000))
# Ensamblaje: el mejor bosque de la Semana 10, con hojas de al menos 2 discos.
evaluar("bosque min_samples_leaf=2", RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))
# Red neuronal simple: dos capas ocultas; early_stopping aparta 10 % del
# entrenamiento y para cuando deja de mejorar.
evaluar("red neuronal 32-16", MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=300, random_state=42))

linea base                   entrenamiento 0.0594   prueba 0.0594      0.7 s


logistica C=1                entrenamiento 0.2410   prueba 0.2214      2.4 s


bosque min_samples_leaf=2    entrenamiento 0.7477   prueba 0.5003     12.5 s


red neuronal 32-16           entrenamiento 0.3919   prueba 0.3593     35.1 s


## La corrida final, cargada desde el registro

In [6]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
tabla = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    filter_string=f"tags.ejecucion = '{ejecucion}'",
    order_by=["metrics.ap_prueba DESC"],
)
final = tabla.iloc[0]
mlflow.MlflowClient().set_tag(final["run_id"], "candidato", "final")
print("Final:", final["tags.mlflow.runName"], " run_id:", final["run_id"], f" ap_prueba {final['metrics.ap_prueba']:.4f}")

Final: bosque min_samples_leaf=2  run_id: 2a4bbdc38c794ba1a7955c6493234e43  ap_prueba 0.5003


In [7]:
# Reproducir: cargar el pipeline guardado y medir otra vez sobre la misma prueba.
cargado = mlflow.sklearn.load_model(f"runs:/{final['run_id']}/modelo")
ap_cargado = average_precision_score(y_prueba, cargado.predict_proba(X_prueba)[:, 1])
print(f"Registrada {final['metrics.ap_prueba']:.6f}   cargada {ap_cargado:.6f}")
assert np.isclose(ap_cargado, final["metrics.ap_prueba"]), "El modelo cargado no reproduce la metrica registrada"
print("El modelo del registro reproduce la metrica.")

Registrada 0.500256   cargada 0.500256
El modelo del registro reproduce la metrica.


## La tabla de corridas

In [8]:
tabla = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    filter_string=f"tags.ejecucion = '{ejecucion}'",
    order_by=["metrics.ap_prueba DESC"],
)
tabla.to_csv("corridas_proyecto.csv", index=False)
columnas = ["run_id", "tags.mlflow.runName", "tags.candidato", "params.huella_datos", "params.semilla_division",
            "metrics.ap_entrenamiento", "metrics.ap_prueba", "metrics.auc_prueba", "metrics.segundos"]
print(tabla[[c for c in columnas if c in tabla.columns]].round(4).to_string(index=False))

                          run_id       tags.mlflow.runName tags.candidato params.huella_datos params.semilla_division  metrics.ap_entrenamiento  metrics.ap_prueba  metrics.auc_prueba  metrics.segundos
2a4bbdc38c794ba1a7955c6493234e43 bosque min_samples_leaf=2          final        ef33432c8970                      42                    0.7477             0.5003              0.8619           12.5072
ee56ada9b26f4cc6a7fc29beda83952d        red neuronal 32-16            NaN        ef33432c8970                      42                    0.3919             0.3593              0.8102           35.1301
0d215d9e713446ea8e52888ea50f9310             logistica C=1            NaN        ef33432c8970                      42                    0.2410             0.2214              0.7195            2.4124
dab60cde549f4b2db6567a81f8a6b996                linea base            NaN        ef33432c8970                      42                    0.0594             0.0594              0.5000            0.